In [ ]:
import os
import requests

# f.i.juraev@gmail.com 
API_KEY = os.getenv("IMAGGA_API_KEY", "")
API_SECRET = os.getenv("IMAGGA_API_SECRET", "")
IMAGE_PATH = "google_test_images/adversarial/img_0000.png"


# Upload image
with open(IMAGE_PATH, "rb") as image_file:
    upload_response = requests.post(
        "https://api.imagga.com/v2/uploads",
        auth=(API_KEY, API_SECRET),
        files={"image": image_file},
        timeout=60
    )

upload_response.raise_for_status()

upload_id = upload_response.json()["result"]["upload_id"]

print("Upload successful")
print("Upload ID:", upload_id)


# Get tags
tag_response = requests.get(
    "https://api.imagga.com/v2/tags",
    auth=(API_KEY, API_SECRET),
    params={
        "image_upload_id": upload_id
    },
    timeout=60
)

tag_response.raise_for_status()

result = tag_response.json()


# Print top 10 predictions
print("\nTop predictions:\n")

tags = result["result"]["tags"][:10]

for i, item in enumerate(tags, start=1):
    label = item["tag"]["en"]
    confidence = item["confidence"]

    print(f"{i:2d}. {label:25s} {confidence:.2f}%")

In [ ]:
import os
import base64
import requests
import pandas as pd
from tqdm import tqdm

API_KEY = os.getenv("IMAGGA_API_KEY", "")
API_SECRET = os.getenv("IMAGGA_API_SECRET", "")
IMAGE_FOLDER = r"google_test_images\benign"
OUTPUT_CSV = "imagga_benign_results.csv"

TOP_K = 10
MAX_IMAGES = 75   # stay within daily quota


def get_tags(image_path):

    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")

    response = requests.post(
        "https://api.imagga.com/v2/tags",
        auth=(API_KEY, API_SECRET),
        data={
            "image_base64": image_base64
        },
        timeout=60
    )

    response.raise_for_status()

    return response.json()["result"]["tags"][:TOP_K]


extensions = (".png", ".jpg", ".jpeg", ".webp")

images = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith(extensions)
])

# Only process first 100 today
images = images[:MAX_IMAGES]

print("Processing:", len(images), "images")

rows = []

for filename in tqdm(images):

    path = os.path.join(IMAGE_FOLDER, filename)

    try:
        tags = get_tags(path)

        row = {
            "image": filename,
            "status": "success"
        }

        for i, tag in enumerate(tags, start=1):
            row[f"top{i}_label"] = tag["tag"]["en"]
            row[f"top{i}_confidence"] = tag["confidence"]

    except Exception as e:

        row = {
            "image": filename,
            "status": "failed",
            "error": str(e)
        }

    rows.append(row)


df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("\nSaved:", OUTPUT_CSV)

In [ ]:
df

In [ ]:
df

In [ ]:
import pandas as pd

BENIGN_CSV = "imagga_benign_results.csv"
ADV_CSV = "imagga_adversarial_results.csv"

TOP_K = 5

benign = pd.read_csv(BENIGN_CSV)
adv = pd.read_csv(ADV_CSV)

# Keep only successful queries
benign = benign[benign["status"] == "success"].copy()
adv = adv[adv["status"] == "success"].copy()

# Merge same images
df = benign.merge(
    adv,
    on="image",
    suffixes=("_benign", "_adv")
)

print("Paired images:", len(df))


# ============================================================
# 1. TOP-1 LABEL CHANGE
# ============================================================

df["top1_changed"] = (
    df["top1_label_benign"].str.lower()
    !=
    df["top1_label_adv"].str.lower()
)

top1_change_rate = df["top1_changed"].mean() * 100


# ============================================================
# 2. TOP-1 CONFIDENCE DROP
# ============================================================

df["top1_conf_drop"] = (
    df["top1_confidence_benign"]
    -
    df["top1_confidence_adv"]
)

mean_conf_drop = df["top1_conf_drop"].mean()


# ============================================================
# 3. TOP-K LABEL OVERLAP
# ============================================================

def get_topk(row, suffix, k):

    labels = []

    for i in range(1, k + 1):

        col = f"top{i}_label_{suffix}"

        if col in row and pd.notna(row[col]):
            labels.append(
                str(row[col]).strip().lower()
            )

    return set(labels)


def overlap(row):

    benign_labels = get_topk(
        row,
        "benign",
        TOP_K
    )

    adv_labels = get_topk(
        row,
        "adv",
        TOP_K
    )

    if len(benign_labels) == 0:
        return 0

    return len(
        benign_labels & adv_labels
    ) / len(benign_labels)


df[f"top{TOP_K}_overlap"] = df.apply(
    overlap,
    axis=1
)

mean_overlap = (
    df[f"top{TOP_K}_overlap"].mean()
    * 100
)


# ============================================================
# 4. COMPLETE TOP-K DISRUPTION
# ============================================================

df["complete_disruption"] = (
    df[f"top{TOP_K}_overlap"] == 0
)

complete_disruption_rate = (
    df["complete_disruption"].mean()
    * 100
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n====================================")
print("IMAGGA ADVERSARIAL EVALUATION")
print("====================================")

print(
    f"Number of paired images: {len(df)}"
)

print(
    f"Top-1 Label Change Rate: "
    f"{top1_change_rate:.2f}%"
)

print(
    f"Mean Top-1 Confidence Drop: "
    f"{mean_conf_drop:.2f}"
)

print(
    f"Top-{TOP_K} Label Retention: "
    f"{mean_overlap:.2f}%"
)

print(
    f"Top-{TOP_K} Complete Disruption: "
    f"{complete_disruption_rate:.2f}%"
)


# ============================================================
# SAVE DETAILED RESULTS
# ============================================================

df.to_csv(
    "imagga_pairwise_evaluation.csv",
    index=False
)

print(
    "\nSaved: imagga_pairwise_evaluation.csv"
)

In [ ]:
# ============================================================
# CLEAN DUPLICATES
# ============================================================

def clean_api_results(df, name="results"):
    df = df.copy()

    print(f"\n{name} before cleaning: {len(df)} rows")

    # Prefer successful API calls over failed ones
    if "status" in df.columns:
        df["_success_priority"] = (
            df["status"]
            .fillna("")
            .str.lower()
            .eq("success")
            .astype(int)
        )

        # Put successful rows first for each image
        df = df.sort_values(
            by=["image", "_success_priority"],
            ascending=[True, False]
        )

    # Keep only one row per image
    df = df.drop_duplicates(
        subset=["image"],
        keep="first"
    ).copy()

    # Remove temporary helper column
    if "_success_priority" in df.columns:
        df = df.drop(columns=["_success_priority"])

    print(f"{name} after cleaning:  {len(df)} rows")
    print(
        f"{name} remaining duplicate images:",
        df["image"].duplicated().sum()
    )

    return df


benign = clean_api_results(
    benign,
    "Benign"
)

adv = clean_api_results(
    adv,
    "Adversarial"
)

In [ ]:
BENIGN_CSV = "imagga_benign_results_1.csv"
ADV_CSV = "imagga_adversarial_results_1.csv"

benign.to_csv(BENIGN_CSV, index=False)
adv.to_csv(ADV_CSV, index=False)

In [ ]:
import pandas as pd
import re


# ============================================================
# FILES
# ============================================================

LABELS_CSV = "google_test_images/labels.csv"

BENIGN_CSV = "imagga_benign_results_1.csv"
# ADV_CSV = "imagga_adversarial_results_MI_FGSM_SMER.csv"
ADV_CSV = "imagga_adversarial_results_SMER_ViT.csv"

OUTPUT_CSV = "imagga_evaluation_detailed.csv"


# ============================================================
# LOAD
# ============================================================

labels = pd.read_csv(LABELS_CSV)
benign = pd.read_csv(BENIGN_CSV)
adv = pd.read_csv(ADV_CSV)


# ============================================================
# MERGE WITH GROUND TRUTH
# ============================================================

df = labels.merge(
    benign,
    left_on="benign_image",
    right_on="image",
    how="inner"
)

df = df.merge(
    adv,
    left_on="adversarial_image",
    right_on="image",
    how="inner",
    suffixes=("_benign", "_adv")
)

print("Number of paired images:", len(df))


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = text.replace("-", " ")
    text = text.replace("_", " ")

    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text


# ============================================================
# SEMANTIC STRING MATCH
# ============================================================
#
# Examples:
#
# GT: hammerhead
# API: hammerhead shark
# -> correct
#
# GT: tree frog
# API: frog
# -> correct
#
# GT: king snake
# API: snake
# -> correct
#
# This is more reasonable for generic tagging APIs than
# requiring exact ImageNet class names.
# ============================================================

def semantic_match(gt, prediction):
    gt = normalize(gt)
    prediction = normalize(prediction)

    if not gt or not prediction:
        return False

    # Exact match
    if gt == prediction:
        return True

    # One phrase contained inside the other
    if gt in prediction or prediction in gt:
        return True

    gt_words = set(gt.split())
    pred_words = set(prediction.split())

    # Shared meaningful word
    common = gt_words.intersection(pred_words)

    if len(common) > 0:
        return True

    return False


# ============================================================
# CHECK TOP-K
# ============================================================

def correct_at_k(row, suffix, k):
    gt = row["label_name"]

    for i in range(1, k + 1):
        column = f"top{i}_label_{suffix}"
        if column not in row.index:
            continue

        prediction = row[column]

        if semantic_match(gt, prediction):
            return True

    return False


# ============================================================
# CALCULATE CORRECTNESS
# ============================================================

for k in [1, 5, 10]:

    df[f"benign_top{k}_correct"] = df.apply(
        lambda row: correct_at_k(
            row,
            "benign",
            k
        ),
        axis=1
    )

    df[f"adv_top{k}_correct"] = df.apply(
        lambda row: correct_at_k(
            row,
            "adv",
            k
        ),
        axis=1
    )


# ============================================================
# ACCURACY
# ============================================================

print("\n============================================")
print("        IMAGGA ROBUSTNESS RESULTS")
print("============================================")

for k in [1, 5, 10]:

    benign_acc = (
        df[f"benign_top{k}_correct"].mean()
        * 100
    )

    adv_acc = (df[f"adv_top{k}_correct"].mean() * 100)

    drop = benign_acc - adv_acc

    print(f"\nTOP-{k}")

    print(f"Benign accuracy:       " f"{benign_acc:.2f}%")

    print(
        f"Adversarial accuracy:  "
        f"{adv_acc:.2f}%"
    )

    print(
        f"Accuracy drop:         "
        f"{drop:.2f}%"
    )


# ============================================================
# ATTACK SUCCESS RATE
# ============================================================
#
# ASR is calculated ONLY on images that were correctly
# recognized before attack:
#
#        Correct benign -> Incorrect adversarial
#
# ============================================================

print("\n============================================")
print("             ATTACK SUCCESS RATE")
print("============================================")

for k in [1, 5, 10]:

    benign_correct = df[
        df[f"benign_top{k}_correct"]
    ]

    attacked_successfully = benign_correct[
        ~benign_correct[f"adv_top{k}_correct"]
    ]

    if len(benign_correct) > 0:

        asr = (
            len(attacked_successfully)
            /
            len(benign_correct)
            * 100
        )

    else:
        asr = 0

    print(f"\nTop-{k}:")

    print(
        "Benign correct:",
        len(benign_correct)
    )

    print(
        "Successfully attacked:",
        len(attacked_successfully)
    )

    print(
        f"ASR: {asr:.2f}%"
    )


# ============================================================
# TOP-1 PREDICTION CHANGE RATE
# ============================================================

df["top1_changed"] = (
    df["top1_label_benign"]
    .fillna("")
    .apply(normalize)
    !=
    df["top1_label_adv"]
    .fillna("")
    .apply(normalize)
)

change_rate = (
    df["top1_changed"].mean()
    * 100
)

print("\n============================================")
print("Top-1 prediction change rate:")
print(f"{change_rate:.2f}%")
print("============================================")


# ============================================================
# SAVE DETAILED RESULTS
# ============================================================

columns_to_save = [
    "image_id",
    "benign_image",
    "adversarial_image",
    "label_id",
    "label_name",

    "top1_label_benign",
    "top1_confidence_benign",

    "top1_label_adv",
    "top1_confidence_adv",

    "benign_top1_correct",
    "adv_top1_correct",

    "benign_top5_correct",
    "adv_top5_correct",

    "benign_top10_correct",
    "adv_top10_correct",

    "top1_changed"
]

available_columns = [
    c for c in columns_to_save
    if c in df.columns
]

df[available_columns].to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nDetailed results saved to:")
print(OUTPUT_CSV)

### Append 

In [ ]:
import os
import base64
import requests
import pandas as pd
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================
API_KEY = os.getenv("IMAGGA_API_KEY", "")
API_SECRET = os.getenv("IMAGGA_API_SECRET", "")
# IMAGE_FOLDER = r"google_test_images\benign"
# OUTPUT_CSV = "imagga_benign_results.csv"

IMAGE_FOLDER = r"google_test_images\SMER_ViT"
OUTPUT_CSV = "imagga_adversarial_results_SMER_ViT.csv"




TOP_K = 10

# Number of NEW images to process this run
MAX_IMAGES = 100


# ============================================================
# IMAGGA
# ============================================================

def get_tags(image_path):

    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(
            f.read()
        ).decode("utf-8")

    response = requests.post(
        "https://api.imagga.com/v2/tags",
        auth=(API_KEY, API_SECRET),
        data={
            "image_base64": image_base64
        },
        timeout=60
    )

    response.raise_for_status()

    return response.json()["result"]["tags"][:TOP_K]


# ============================================================
# FIND ALL IMAGES
# ============================================================

extensions = (".png", ".jpg", ".jpeg", ".webp")

all_images = sorted([f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith(extensions)])

# ============================================================
# READ EXISTING RESULTS
# ============================================================

processed_successfully = set()

if os.path.exists(OUTPUT_CSV):
    existing_df = pd.read_csv(OUTPUT_CSV)
    if "status" in existing_df.columns:
        successful_df = existing_df[
            existing_df["status"] == "success"
        ]

        processed_successfully = set(
            successful_df["image"]
            .astype(str)
            .tolist()
        )

    else:
        processed_successfully = set(existing_df["image"].astype(str).tolist())


# ============================================================
# GET ONLY UNPROCESSED IMAGES
# ============================================================

remaining_images = [
    filename
    for filename in all_images
    if filename not in processed_successfully
]


# Only process the NEXT batch
images_to_process = remaining_images[:MAX_IMAGES]


print("=" * 60)
print("IMAGGA BATCH PROCESSING")
print("=" * 60)

print("Total images:", len(all_images))
print("Already processed successfully:", len(processed_successfully))
print("Still remaining:", len(remaining_images))
print("Processing this run:", len(images_to_process))

print("=" * 60)


# ============================================================
# PROCESS
# ============================================================

for filename in tqdm(images_to_process, desc="Imagga"):
    path = os.path.join(IMAGE_FOLDER, filename)

    row = {"image": filename}

    try:

        tags = get_tags(path)

        row["status"] = "success"

        for i, tag in enumerate(
            tags,
            start=1
        ):

            row[f"top{i}_label"] = (
                tag["tag"]["en"]
            )

            row[f"top{i}_confidence"] = (
                tag["confidence"]
            )

    except Exception as e:

        row["status"] = "failed"
        row["error"] = str(e)

        print(
            f"\nFailed: {filename}"
        )

        print(e)


    # ========================================================
    # APPEND IMMEDIATELY
    # ========================================================

    new_row = pd.DataFrame([row])


    if os.path.exists(OUTPUT_CSV):

        new_row.to_csv(
            OUTPUT_CSV,
            mode="a",
            header=False,
            index=False
        )

    else:

        new_row.to_csv(
            OUTPUT_CSV,
            mode="w",
            header=True,
            index=False
        )


# ============================================================
# FINAL STATUS
# ============================================================

final_df = pd.read_csv(OUTPUT_CSV)
success_count = (final_df["status"] == "success").sum()
failed_count = (final_df["status"] == "failed").sum()


print("\n" + "=" * 60)
print("FINISHED")
print("=" * 60)

print("Total successful records:", success_count)

print("Failed records:", failed_count)
print("Images still remaining:", len(all_images) - success_count)
print("Saved:", OUTPUT_CSV)